# GPU Assignment Testbed

This notebook is a small testbed for `DisaggInferenceSession._build_gpu_layout()`.
It uses the real layout builder and prints:

- `gpu_ids`
- `pp_stages`
- the implied TP-rank to GPU mapping inside each PP stage
- `first_stage_gpus` and `last_stage_gpus`

This is useful for understanding how TP/PP worker layouts are turned into GPU ID assignments before AstraSim classifies links.


In [1]:
from types import SimpleNamespace

from aiconfigurator.sdk.inference_session import DisaggInferenceSession
from aiconfigurator.sdk import config
from aiconfigurator.sdk.backends.factory import get_backend
from aiconfigurator.sdk.perf_database import (
    PerfDatabase,
    get_latest_database_version,
    get_supported_databases,
    get_system_config_path,
)


def make_model_config(tp: int, pp: int) -> config.ModelConfig:
    return config.ModelConfig(
        tp_size=tp,
        pp_size=pp,
        moe_tp_size=1,
        moe_ep_size=1,
        attention_dp_size=1,
    )


def make_attention_model_config(tp: int, pp: int, num_attn_gpus: int | None = None) -> config.ModelConfig:
    mc = make_model_config(tp, pp)
    mc.enable_afd = True
    mc.num_attn_gpus = num_attn_gpus if num_attn_gpus is not None else tp * pp
    return mc


def make_ffn_model_config(tp: int, pp: int, num_ffn_gpus: int | None = None) -> config.ModelConfig:
    mc = make_model_config(tp, pp)
    mc.enable_afd = True
    mc.num_ffn_gpus = num_ffn_gpus if num_ffn_gpus is not None else tp * pp
    return mc


def make_runtime_config(
    batch_size: int = 1,
    isl: int = 4000,
    osl: int = 500,
    beam_width: int = 1,
    prefix: int = 0,
) -> config.RuntimeConfig:
    return config.RuntimeConfig(
        batch_size=batch_size,
        isl=isl,
        osl=osl,
        beam_width=beam_width,
        prefix=prefix,
    )


def get_default_version(system: str, backend_name: str) -> str:
    version = get_latest_database_version(system, backend_name)
    if version is None:
        supported = get_supported_databases()
        raise ValueError(
            f"No database version found for system={system!r}, backend={backend_name!r}. "
            f"Supported entries: {dict(supported.get(system, {}))}"
        )
    return version


def make_perf_database(system: str, backend_name: str = "trtllm", version: str | None = None) -> PerfDatabase:
    systems_dir = get_system_config_path()
    resolved_version = version or get_default_version(system, backend_name)
    return PerfDatabase(system, backend_name, resolved_version, systems_dir=str(systems_dir))


def make_real_disagg_session(
    system: str,
    backend_name: str = "trtllm",
    version: str | None = None,
    network_file: str | None = None,
    gpu_layout_strategy: str = "segregated_by_phase",
) -> DisaggInferenceSession:
    database = make_perf_database(system=system, backend_name=backend_name, version=version)
    backend = get_backend(backend_name)
    return DisaggInferenceSession(
        prefill_database=database,
        prefill_backend=backend,
        decode_database=database,
        decode_backend=backend,
        network_file=network_file,
        gpu_layout_strategy=gpu_layout_strategy,
    )


def run_disagg_case(
    session: DisaggInferenceSession,
    model_path: str,
    runtime_config: config.RuntimeConfig,
    prefill_model_config: config.ModelConfig,
    prefill_batch_size: int,
    prefill_num_worker: int,
    decode_model_config: config.ModelConfig,
    decode_batch_size: int,
    decode_num_worker: int,
):
    summary = session.run_disagg(
        model_path=model_path,
        runtime_config=runtime_config,
        prefill_model_config=prefill_model_config,
        prefill_batch_size=prefill_batch_size,
        prefill_num_worker=prefill_num_worker,
        decode_model_config=decode_model_config,
        decode_batch_size=decode_batch_size,
        decode_num_worker=decode_num_worker,
    )

    df = summary.get_summary_df()
    net = summary.get_network_info()
    row = df.iloc[0]

    print("=" * 90)
    print(f"model_path                  = {model_path}")
    print(
        f"runtime_config              = batch={runtime_config.batch_size}, isl={runtime_config.isl}, "
        f"osl={runtime_config.osl}, beam={runtime_config.beam_width}, prefix={runtime_config.prefix}"
    )
    print(
        f"prefill                     = TP={prefill_model_config.tp_size}, PP={prefill_model_config.pp_size}, "
        f"workers={prefill_num_worker}, batch={prefill_batch_size}"
    )
    print(
        f"decode                      = TP={decode_model_config.tp_size}, PP={decode_model_config.pp_size}, "
        f"workers={decode_num_worker}, batch={decode_batch_size}"
    )
    print(f"gpu_layout_strategy         = {net.get('gpu_layout', {}).get('gpu_layout_strategy')}")
    print(f"kv_cache_size_bytes         = {net.get('kv_cache_size_bytes', 0):,}")
    print(f"kv_cache_size_mb            = {net.get('kv_cache_size_bytes', 0) / 1e6:.3f}")
    print(f"kv_network_latency_ms       = {net.get('kv_network_latency_ms', 0.0):.6f}")
    print(f"kv_transfer_pct_of_ttft     = {summary.get_kv_cache_transfer_pct():.3f}%")
    print("-")
    print(f"ttft                        = {float(row['ttft']):.6f} ms")
    print(f"tpot                        = {float(row['tpot']):.6f} ms")
    print(f"request_latency             = {float(row['request_latency']):.6f} ms")
    print(f"tokens/s                    = {float(row['tokens/s']):.6f}")
    print(f"tokens/s/gpu                = {float(row['tokens/s/gpu']):.6f}")
    print("=" * 90)
    return summary


def make_layout_session(
    gpu_layout_strategy: str = "segregated_by_phase",
    num_gpus_per_node: int = 8,
) -> DisaggInferenceSession:
    """Create a minimal session object that can call _build_gpu_layout()."""
    sess = DisaggInferenceSession.__new__(DisaggInferenceSession)
    sess._gpu_layout_strategy = gpu_layout_strategy
    sess._prefill_database = SimpleNamespace(
        system_spec={"node": {"num_gpus_per_node": num_gpus_per_node}}
    )
    return sess


def print_worker_layout(role: str, worker: dict) -> None:
    print(f"{role}[{worker['worker_id']}]")
    print(f"  gpu_ids          = {worker['gpu_ids']}")
    print(f"  pp_stages        = {worker['pp_stages']}")
    for pp_rank, stage in enumerate(worker['pp_stages']):
        tp_map = {f'tp{tp_rank}': gpu_id for tp_rank, gpu_id in enumerate(stage)}
        print(f"  PP{pp_rank} TP map   = {tp_map}")
    print(f"  first_stage_gpus = {worker['first_stage_gpus']}")
    print(f"  last_stage_gpus  = {worker['last_stage_gpus']}")
    print()


def show_named_pair_layout_from_configs(
    left_label: str,
    left_model_config: config.ModelConfig,
    left_workers: int,
    right_label: str,
    right_model_config: config.ModelConfig,
    right_workers: int,
    strategy: str = "segregated_by_phase",
    num_gpus_per_node: int = 8,
) -> dict:
    sess = make_layout_session(strategy, num_gpus_per_node)
    layout = sess._build_gpu_layout(
        prefill_model_config=left_model_config,
        prefill_num_worker=left_workers,
        decode_model_config=right_model_config,
        decode_num_worker=right_workers,
    )

    print("=" * 90)
    print(
        f"strategy={strategy} | "
        f"{left_label}: TP={left_model_config.tp_size}, PP={left_model_config.pp_size}, workers={left_workers} | "
        f"{right_label}: TP={right_model_config.tp_size}, PP={right_model_config.pp_size}, workers={right_workers}"
    )
    print("-" * 90)
    print(f"prefill_workers({left_label})  = {layout['prefill_workers']}")
    print(f"decode_workers({right_label})  = {layout['decode_workers']}")
    print(f"gpu_layout_strategy            = {layout['gpu_layout_strategy']}")
    print(f"max_gpu_id_plus_one            = {layout['max_gpu_id_plus_one']}")
    print(f"total_prefill_gpus             = {layout['total_prefill_gpus']}")
    print(f"total_decode_gpus              = {layout['total_decode_gpus']}")
    print()

    for worker in layout['prefill_worker_layouts']:
        print_worker_layout(left_label, worker)
    for worker in layout['decode_worker_layouts']:
        print_worker_layout(right_label, worker)

    return layout


def show_layout_from_configs(
    prefill_model_config: config.ModelConfig,
    prefill_workers: int,
    decode_model_config: config.ModelConfig,
    decode_workers: int,
    strategy: str = "segregated_by_phase",
    num_gpus_per_node: int = 8,
) -> dict:
    return show_named_pair_layout_from_configs(
        "Prefill",
        prefill_model_config,
        prefill_workers,
        "Decode",
        decode_model_config,
        decode_workers,
        strategy=strategy,
        num_gpus_per_node=num_gpus_per_node,
    )


def show_layout(
    prefill_tp: int,
    prefill_pp: int,
    prefill_workers: int,
    decode_tp: int,
    decode_pp: int,
    decode_workers: int,
    strategy: str = "segregated_by_phase",
    num_gpus_per_node: int = 8,
) -> dict:
    return show_layout_from_configs(
        prefill_model_config=make_model_config(prefill_tp, prefill_pp),
        prefill_workers=prefill_workers,
        decode_model_config=make_model_config(decode_tp, decode_pp),
        decode_workers=decode_workers,
        strategy=strategy,
        num_gpus_per_node=num_gpus_per_node,
    )


## Example 1: TP=2, PP=2 for both prefill and decode

This is the exact case discussed in the thread.
With the default `segregated_by_phase` strategy, you should see:

- Prefill worker: `pp_stages = [[0, 1], [2, 3]]`
- Decode worker: `pp_stages = [[4, 5], [6, 7]]`


In [2]:
prefill_model_config = make_model_config(tp=2, pp=2)
decode_model_config = make_model_config(tp=2, pp=2)

print("prefill_model_config =", prefill_model_config)
print("decode_model_config  =", decode_model_config)
print()

layout_tp2_pp2 = show_layout_from_configs(
    prefill_model_config=prefill_model_config,
    prefill_workers=1,
    decode_model_config=decode_model_config,
    decode_workers=1,
    strategy="segregated_by_phase",
)



prefill_model_config = ModelConfig(tp_size=2, pp_size=2, gemm_quant_mode=<GEMMQuantMode.float16: QuantMapping(memory=2, compute=1, name='float16')>, moe_quant_mode=<MoEQuantMode.float16: QuantMapping(memory=2, compute=1, name='float16')>, kvcache_quant_mode=<KVCacheQuantMode.float16: QuantMapping(memory=2, compute=0, name='float16')>, fmha_quant_mode=<FMHAQuantMode.float16: QuantMapping(memory=2, compute=1, name='float16')>, comm_quant_mode=<CommQuantMode.half: QuantMapping(memory=2, compute=0, name='half')>, moe_tp_size=1, moe_ep_size=1, attention_dp_size=1, workload_distribution='power_law', nextn=0, nextn_accept_rates=None, overwrite_num_layers=0, sms=20, moe_backend=None, attention_backend='flashinfer', enable_wideep=False, enable_afd=False, num_attn_gpus=None, num_ffn_gpus=None, afd_num_microbatches=1)
decode_model_config  = ModelConfig(tp_size=2, pp_size=2, gemm_quant_mode=<GEMMQuantMode.float16: QuantMapping(memory=2, compute=1, name='float16')>, moe_quant_mode=<MoEQuantMode.flo

## Example 2: 2 prefill workers and 2 decode workers packed as 1P+1D per node

This uses the placement strategy `paired_prefill_decode_per_node`.
If the node has 8 GPUs and each worker uses 1 GPU, the modeled placement becomes:

- node 0: `P0`, `D0`
- node 1: `P1`, `D1`


In [3]:
prefill_model_config = make_model_config(tp=1, pp=1)
decode_model_config = make_model_config(tp=1, pp=1)

print("prefill_model_config =", prefill_model_config)
print("decode_model_config  =", decode_model_config)
print()

layout_2p2d_paired = show_layout_from_configs(
    prefill_model_config=prefill_model_config,
    prefill_workers=2,
    decode_model_config=decode_model_config,
    decode_workers=2,
    strategy="paired_prefill_decode_per_node",
    num_gpus_per_node=8,
)



prefill_model_config = ModelConfig(tp_size=1, pp_size=1, gemm_quant_mode=<GEMMQuantMode.float16: QuantMapping(memory=2, compute=1, name='float16')>, moe_quant_mode=<MoEQuantMode.float16: QuantMapping(memory=2, compute=1, name='float16')>, kvcache_quant_mode=<KVCacheQuantMode.float16: QuantMapping(memory=2, compute=0, name='float16')>, fmha_quant_mode=<FMHAQuantMode.float16: QuantMapping(memory=2, compute=1, name='float16')>, comm_quant_mode=<CommQuantMode.half: QuantMapping(memory=2, compute=0, name='half')>, moe_tp_size=1, moe_ep_size=1, attention_dp_size=1, workload_distribution='power_law', nextn=0, nextn_accept_rates=None, overwrite_num_layers=0, sms=20, moe_backend=None, attention_backend='flashinfer', enable_wideep=False, enable_afd=False, num_attn_gpus=None, num_ffn_gpus=None, afd_num_microbatches=1)
decode_model_config  = ModelConfig(tp_size=1, pp_size=1, gemm_quant_mode=<GEMMQuantMode.float16: QuantMapping(memory=2, compute=1, name='float16')>, moe_quant_mode=<MoEQuantMode.flo

## Example 3: A few more layouts to inspect

Feel free to edit these and rerun.


In [4]:
prefill_model_config = make_model_config(tp=2, pp=1)
decode_model_config = make_model_config(tp=4, pp=1)
_ = show_layout_from_configs(prefill_model_config, 1, decode_model_config, 1, strategy="segregated_by_phase")

prefill_model_config = make_model_config(tp=2, pp=2)
decode_model_config = make_model_config(tp=2, pp=2)
_ = show_layout_from_configs(prefill_model_config, 2, decode_model_config, 2, strategy="segregated_by_phase")

prefill_model_config = make_model_config(tp=2, pp=1)
decode_model_config = make_model_config(tp=2, pp=1)
_ = show_layout_from_configs(
    prefill_model_config,
    2,
    decode_model_config,
    2,
    strategy="paired_prefill_decode_per_node",
    num_gpus_per_node=8,
)



strategy=segregated_by_phase | Prefill: TP=2, PP=1, workers=1 | Decode: TP=4, PP=1, workers=1
------------------------------------------------------------------------------------------
prefill_workers(Prefill)  = [[0, 1]]
decode_workers(Decode)  = [[2, 3, 4, 5]]
gpu_layout_strategy            = segregated_by_phase
max_gpu_id_plus_one            = 6
total_prefill_gpus             = 2
total_decode_gpus              = 4

Prefill[0]
  gpu_ids          = [0, 1]
  pp_stages        = [[0, 1]]
  PP0 TP map   = {'tp0': 0, 'tp1': 1}
  first_stage_gpus = [0, 1]
  last_stage_gpus  = [0, 1]

Decode[0]
  gpu_ids          = [2, 3, 4, 5]
  pp_stages        = [[2, 3, 4, 5]]
  PP0 TP map   = {'tp0': 2, 'tp1': 3, 'tp2': 4, 'tp3': 5}
  first_stage_gpus = [2, 3, 4, 5]
  last_stage_gpus  = [2, 3, 4, 5]

strategy=segregated_by_phase | Prefill: TP=2, PP=2, workers=2 | Decode: TP=2, PP=2, workers=2
------------------------------------------------------------------------------------------
prefill_workers(Prefil

## Example 4: AFD-style Attention / FFN group configs

Here we create separate `attention_model_config` and `ffn_model_config` objects.
This is useful for placement experiments where you want to inspect the two groups
separately instead of inheriting prefill/decode configs.

Note: the current `_build_gpu_layout()` API still uses one shared `system_spec`
per call, so this notebook can separate the config objects and role labels, but
it does not yet model two different system specs at the same time inside one
layout call.



In [5]:
attention_model_config = make_attention_model_config(tp=2, pp=1, num_attn_gpus=2)
ffn_model_config = make_ffn_model_config(tp=4, pp=1, num_ffn_gpus=4)

print("attention_model_config =", attention_model_config)
print("ffn_model_config       =", ffn_model_config)
print()

afd_layout = show_named_pair_layout_from_configs(
    left_label="Attention",
    left_model_config=attention_model_config,
    left_workers=1,
    right_label="FFN",
    right_model_config=ffn_model_config,
    right_workers=1,
    strategy="segregated_by_phase",
    num_gpus_per_node=8,
)



attention_model_config = ModelConfig(tp_size=2, pp_size=1, gemm_quant_mode=<GEMMQuantMode.float16: QuantMapping(memory=2, compute=1, name='float16')>, moe_quant_mode=<MoEQuantMode.float16: QuantMapping(memory=2, compute=1, name='float16')>, kvcache_quant_mode=<KVCacheQuantMode.float16: QuantMapping(memory=2, compute=0, name='float16')>, fmha_quant_mode=<FMHAQuantMode.float16: QuantMapping(memory=2, compute=1, name='float16')>, comm_quant_mode=<CommQuantMode.half: QuantMapping(memory=2, compute=0, name='half')>, moe_tp_size=1, moe_ep_size=1, attention_dp_size=1, workload_distribution='power_law', nextn=0, nextn_accept_rates=None, overwrite_num_layers=0, sms=20, moe_backend=None, attention_backend='flashinfer', enable_wideep=False, enable_afd=True, num_attn_gpus=2, num_ffn_gpus=None, afd_num_microbatches=1)
ffn_model_config       = ModelConfig(tp_size=4, pp_size=1, gemm_quant_mode=<GEMMQuantMode.float16: QuantMapping(memory=2, compute=1, name='float16')>, moe_quant_mode=<MoEQuantMode.flo

## Example 5: Build a real `DisaggInferenceSession`

This uses the actual constructor with a real `PerfDatabase` and backend, so you can
run `run_disagg(...)` and study TTFT together with KV-cache transfer latency.

Edit `system`, `backend_name`, `version`, `model_path`, runtime config, and the
prefill/decode TP/PP settings to run your own sensitivity sweeps.


In [6]:
system = "b200_sxm"
backend_name = "trtllm"
version = None  # None = auto-pick latest available version for this system/backend

model_path = "Qwen/Qwen3-32B"
runtime_config = make_runtime_config(batch_size=1, isl=4000, osl=500)

prefill_model_config = make_model_config(tp=2, pp=1)
decode_model_config = make_model_config(tp=4, pp=1)

session = make_real_disagg_session(
    system=system,
    backend_name=backend_name,
    version=version,
    gpu_layout_strategy="segregated_by_phase",
)

print("session =", session)
print("prefill_database.system_spec node =", session._prefill_database.system_spec.get("node", {}))
print("prefill_backend =", session._prefill_backend.name.value)
print("decode_backend  =", session._decode_backend.name.value)


INFO:aiconfigurator.sdk.inference_session:AstraSim Network Engine enabled for disagg KV cache transfer


session = <aiconfigurator.sdk.inference_session.DisaggInferenceSession object at 0x7cce04d80610>
prefill_database.system_spec node = {'num_gpus_per_node': 8, 'inter_node_bw': 50000000000, 'intra_node_bw': 900000000000, 'pcie_bw': 128000000000, 'p2p_latency': 1e-05}
prefill_backend = trtllm
decode_backend  = trtllm


## Example 6: Run a real disagg case and inspect TTFT / KV transfer

This cell runs the full session and prints both model performance metrics and the
network-transfer metadata stored in `InferenceSummary`.


In [7]:
summary = run_disagg_case(
    session=session,
    model_path=model_path,
    runtime_config=runtime_config,
    prefill_model_config=prefill_model_config,
    prefill_batch_size=runtime_config.batch_size,
    prefill_num_worker=1,
    decode_model_config=decode_model_config,
    decode_batch_size=max(1, runtime_config.batch_size),
    decode_num_worker=1,
)

summary.get_summary_df()


INFO:aiconfigurator.sdk.utils:Model architecture: architecture=Qwen3ForCausalLM, layers=64, n=64, n_kv=8, d=128, hidden_size=5120, inter_size=25600, vocab=151936, context=40960, topk=0, num_experts=0, moe_inter_size=0, extra_params=None
INFO:aiconfigurator.sdk.inference_session:KV cache transfer size: 1048.58 MB
INFO:aiconfigurator.sdk.inference_session:GPU layout: prefill=[[0, 1]], decode=[[2, 3, 4, 5]]
INFO:aiconfigurator.sdk.astrasim_utils:Created topology config: /scratch1/hanjiang/aiconfigurator/network_backend/astra-network-analytical/input/auto_generated/auto_FullyConnected_6npus_900.0gbps_520ed7cc.yml
INFO:aiconfigurator.sdk.astrasim_utils:AstraSim KV cache transfer (tiered, strategy=tp_overlap_reshard): 0.281 ms for 4 transfers, total 1048576000 bytes
INFO:aiconfigurator.sdk.inference_session:Network transfer latency: 0.281 ms


model_path                  = Qwen/Qwen3-32B
runtime_config              = batch=1, isl=4000, osl=500, beam=1, prefix=0
prefill                     = TP=2, PP=1, workers=1, batch=1
decode                      = TP=4, PP=1, workers=1, batch=1
gpu_layout_strategy         = segregated_by_phase
kv_cache_size_bytes         = 1,048,576,000
kv_cache_size_mb            = 1048.576
kv_network_latency_ms       = 0.281267
kv_transfer_pct_of_ttft     = 0.251%
-
ttft                        = 112.170000 ms
tpot                        = 5.693000 ms
request_latency             = 2952.977000 ms
tokens/s                    = 161.920000
tokens/s/gpu                = 26.987000


,model,isl,osl,prefix,concurrency,request_rate,(p)bs,(p)global_bs,(p)workers,(d)bs,...,(d)fmha,(d)moe,(d)comm,(d)memory,(d)backend,(d)version,(d)system,power_w,kv_network_latency_ms,kv_cache_size_bytes
0,Qwen/Qwen3-32B,4000,500,0,1,0.324,1,1,1,1,...,float16,float16,half,20.568,trtllm,1.2.0rc5,b200_sxm,0.0,0.281267,1048576000


## Example 7: A tiny TTFT sensitivity sweep

This is a convenient starting point for studying how KV transfer changes when you vary
ISL, worker counts, TP ratios, or the GPU layout strategy.


In [8]:
sweep_cases = [
    {
        "label": "same layout, TP 2->4, isl=2048",
        "runtime_config": make_runtime_config(batch_size=1, isl=2048, osl=256),
        "prefill_model_config": make_model_config(tp=2, pp=1),
        "decode_model_config": make_model_config(tp=4, pp=1),
        "prefill_num_worker": 1,
        "decode_num_worker": 1,
        "gpu_layout_strategy": "segregated_by_phase",
    },
    {
        "label": "paired per node, TP 2->4, isl=4096",
        "runtime_config": make_runtime_config(batch_size=1, isl=4096, osl=256),
        "prefill_model_config": make_model_config(tp=2, pp=1),
        "decode_model_config": make_model_config(tp=4, pp=1),
        "prefill_num_worker": 1,
        "decode_num_worker": 1,
        "gpu_layout_strategy": "paired_prefill_decode_per_node",
    },
]

rows = []
for case in sweep_cases:
    case_session = make_real_disagg_session(
        system=system,
        backend_name=backend_name,
        version=version,
        gpu_layout_strategy=case["gpu_layout_strategy"],
    )
    case_summary = case_session.run_disagg(
        model_path=model_path,
        runtime_config=case["runtime_config"],
        prefill_model_config=case["prefill_model_config"],
        prefill_batch_size=case["runtime_config"].batch_size,
        prefill_num_worker=case["prefill_num_worker"],
        decode_model_config=case["decode_model_config"],
        decode_batch_size=max(1, case["runtime_config"].batch_size),
        decode_num_worker=case["decode_num_worker"],
    )
    df = case_summary.get_summary_df()
    net = case_summary.get_network_info()
    rows.append(
        {
            "label": case["label"],
            "layout": case["gpu_layout_strategy"],
            "isl": case["runtime_config"].isl,
            "prefill_tp": case["prefill_model_config"].tp_size,
            "decode_tp": case["decode_model_config"].tp_size,
            "ttft_ms": float(df.iloc[0]["ttft"]),
            "kv_cache_mb": net.get("kv_cache_size_bytes", 0) / 1e6,
            "kv_network_latency_ms": net.get("kv_network_latency_ms", 0.0),
            "kv_pct_of_ttft": case_summary.get_kv_cache_transfer_pct(),
        }
    )

import pandas as pd
pd.DataFrame(rows)


INFO:aiconfigurator.sdk.inference_session:AstraSim Network Engine enabled for disagg KV cache transfer
INFO:aiconfigurator.sdk.inference_session:KV cache transfer size: 536.87 MB
INFO:aiconfigurator.sdk.inference_session:GPU layout: prefill=[[0, 1]], decode=[[2, 3, 4, 5]]
INFO:aiconfigurator.sdk.astrasim_utils:AstraSim KV cache transfer (tiered, strategy=tp_overlap_reshard): 0.149 ms for 4 transfers, total 536870912 bytes
INFO:aiconfigurator.sdk.inference_session:Network transfer latency: 0.149 ms
INFO:aiconfigurator.sdk.inference_session:AstraSim Network Engine enabled for disagg KV cache transfer
INFO:aiconfigurator.sdk.inference_session:KV cache transfer size: 1073.74 MB
INFO:aiconfigurator.sdk.inference_session:GPU layout: prefill=[[0, 1]], decode=[[2, 3, 4, 5]]
INFO:aiconfigurator.sdk.astrasim_utils:AstraSim KV cache transfer (tiered, strategy=tp_overlap_reshard): 0.288 ms for 4 transfers, total 1073741824 bytes
INFO:aiconfigurator.sdk.inference_session:Network transfer latency: 0

,label,layout,isl,prefill_tp,decode_tp,ttft_ms,kv_cache_mb,kv_network_latency_ms,kv_pct_of_ttft
0,"same layout, TP 2->4, isl=2048",segregated_by_phase,2048,2,4,59.442,536.870912,0.148888,0.250476
1,"paired per node, TP 2->4, isl=4096",paired_prefill_decode_per_node,4096,2,4,114.747,1073.741824,0.287777,0.250793
